# Chapter 18: The Sphere for Its Own Sake

Source orientation: printed Volume II pages 255-317; PDF pages 264-326. This notebook is an original visualization-first lesson based on the chapter structure and concepts, not a substitute scan or excerpt.

The chapter question is: how can we turn charts, projections, topology, canonical measure, intrinsic metric, spherical triangles, Clifford parallelism, Villarceau circles, and the Mobius group into objects that can be drawn, measured, transformed, and checked? The answer throughout this notebook is to treat definitions as computational contracts. A convex body becomes hull data and supporting inequalities; a form becomes a symmetric matrix with a visible signature; a conic, sphere, or hyperbolic model becomes an object whose invariants can be probed by code.

The notebook is meant to stand on its own. It introduces the working vocabulary, builds diagrams from scratch, runs small numerical checks, and ends with a lab that can be modified without reopening the book. The source pages are used for orientation: they tell us which ideas belong together, where the chapter puts emphasis, and which examples are worth turning into inspectable experiments.


## Translation guide

- Objects: sphere charts, stereographic projection, great circles, spherical triangles, surface measure, intrinsic distance, and Hopf-style circle families.
- Invariants: unit-length constraints, angular distance, chart distortion, area element behavior, and preservation of circles under stereographic projection.
- Main misconception to disarm: The sphere is not a flat map with curved longitude lines. Its intrinsic geometry lives in angular distance and surface measure, while charts trade local convenience for distortion.
- Computational rule of thumb: start from the algebraic representation, draw the geometric locus, then assert the quantity that should not change.

This translation guide is deliberately practical. It does not try to reproduce every proof. Instead it asks which parts of a proof have a state that can be inspected: an incidence relation, a sign pattern, a limiting family, a deformation, a distance comparison, or a rank calculation. When the theorem is global, the notebook uses examples as probes and labels them as probes. When the claim is an identity, the notebook makes the identity executable.


## Route through the chapter

1. Build a small dictionary of the chapter's objects and the numerical representation used in the notebook.
2. Draw the primary geometric situation with labels and stable axes.
3. Vary a parameter or compare related models so the invariant has something to resist.
4. Save artifacts under `artifacts/chapter-18` and display them inline.
5. Run sanity checks that assert the relevant residuals, distances, signatures, or incidence relations.

The point of this route is not speed. It is to make the reader's eye and the computer agree about what the geometry says. If a diagram is attractive but no invariant is named near it, it is not yet doing mathematical work. If a formula is true but nothing in the notebook lets the reader inspect the objects it relates, the lesson is too thin.


In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the Geometry II book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER = 18
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / f"chapter-{CHAPTER:02d}"
FIGURE_ROOT = ARTIFACT_ROOT / "figures"
PLOT_ROOT = ARTIFACT_ROOT / "plots"
TABLE_ROOT = ARTIFACT_ROOT / "tables"
CHECK_ROOT = ARTIFACT_ROOT / "checks"
for root in [FIGURE_ROOT, PLOT_ROOT, TABLE_ROOT, CHECK_ROOT]:
    root.mkdir(parents=True, exist_ok=True)

print(f"Geometry II root: {BOOK_ROOT}")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from utils.artifacts import assert_artifacts, display_artifact, save_csv, save_json
from utils.geometry import (
    circle_orthogonality,
    conic_matrix,
    conic_residual,
    convex_hull_2d,
    cross_ratio,
    disk_rotation,
    euler_characteristic,
    ellipse_points,
    hyperbola_points,
    polar_line,
    poincare_distance,
    polygon_area,
    quadratic_signature,
    regular_polygon,
    sphere_grid,
    spherical_distance,
    stereographic_project,
)
from utils.plotting import COLORS, finish_axes, new_3d_axes, new_axes, plot_line, plot_points, plot_polyline, plot_unit_circle, save_figure, set_equal_3d

generated_artifacts = []


## Visualization storyboard

- A sphere with great circles paired with a stereographic chart.
- A spherical triangle with angular side lengths.
- A JSON check of unit constraints and spherical distances.

Each visual is paired with a check. The checks are intentionally small and readable: area ratios, matrix signatures, collinearity residuals, distance invariance, and orthogonality errors. This keeps the chapter honest. The plotted object is not a decoration; it is the front end for a reproducible computation.


In [ ]:
x, y, z = sphere_grid(72, 36)
equator = np.column_stack([np.cos(np.linspace(0, 2 * np.pi, 240)), np.sin(np.linspace(0, 2 * np.pi, 240)), np.zeros(240)])
meridian = np.column_stack([np.cos(np.linspace(0, 2 * np.pi, 240)), np.zeros(240), np.sin(np.linspace(0, 2 * np.pi, 240))])
latitude = np.column_stack([0.75 * np.cos(np.linspace(0, 2 * np.pi, 240)), 0.75 * np.sin(np.linspace(0, 2 * np.pi, 240)), np.full(240, math.sqrt(1 - 0.75**2))])

fig = plt.figure(figsize=(12, 5), constrained_layout=True)
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
ax3.plot_surface(x, y, z, color="#dbeafe", alpha=0.55, linewidth=0)
ax3.plot(equator[:, 0], equator[:, 1], equator[:, 2], color=COLORS["blue"], linewidth=2, label="great circle")
ax3.plot(meridian[:, 0], meridian[:, 1], meridian[:, 2], color=COLORS["teal"], linewidth=2, label="great circle")
ax3.plot(latitude[:, 0], latitude[:, 1], latitude[:, 2], color=COLORS["orange"], linewidth=2, label="small circle")
ax3.set_title("sphere with great and small circles")
set_equal_3d(ax3)
ax3.legend(fontsize=8)

ax2 = fig.add_subplot(1, 2, 2)
for curve, color, label in [(equator, COLORS["blue"], "equator"), (meridian[meridian[:, 2] < 0.98], COLORS["teal"], "meridian"), (latitude, COLORS["orange"], "latitude")]:
    projected = stereographic_project(curve)
    ax2.plot(projected[:, 0], projected[:, 1], color=color, linewidth=2, label=label)
ax2.set_aspect("equal")
ax2.grid(True, color="#e5e7eb")
ax2.set_title("stereographic chart")
ax2.legend(fontsize=8)
sphere_path = FIGURE_ROOT / "sphere-stereographic-circles.png"
save_figure(fig, sphere_path)
generated_artifacts.append(sphere_path)
display_artifact(sphere_path)


In [ ]:
triangle = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])
fig, ax = new_3d_axes(title="Spherical triangle with right angular sides")
ax.plot_surface(x, y, z, color="#eef2ff", alpha=0.30, linewidth=0)
for i, j, color in [(0, 1, COLORS["blue"]), (1, 2, COLORS["orange"]), (2, 0, COLORS["teal"])]:
    a = triangle[i]
    b = triangle[j]
    ts = np.linspace(0, 1, 80)
    arc = np.array([(math.sin((1 - t) * math.pi / 2) * a + math.sin(t * math.pi / 2) * b) for t in ts])
    arc = arc / np.linalg.norm(arc, axis=1)[:, None]
    ax.plot(arc[:, 0], arc[:, 1], arc[:, 2], color=color, linewidth=3)
ax.scatter(triangle[:, 0], triangle[:, 1], triangle[:, 2], color=COLORS["red"], s=45)
set_equal_3d(ax)
triangle_path = FIGURE_ROOT / "spherical-triangle-angular-sides.png"
save_figure(fig, triangle_path)
generated_artifacts.append(triangle_path)
display_artifact(triangle_path)


In [ ]:
side_lengths = {
    "AB": spherical_distance(triangle[0], triangle[1]),
    "BC": spherical_distance(triangle[1], triangle[2]),
    "CA": spherical_distance(triangle[2], triangle[0]),
}
checks = {
    "side_lengths": side_lengths,
    "all_sides_pi_over_2": all(abs(value - math.pi / 2) < 1e-12 for value in side_lengths.values()),
    "latitude_unit_residual_max": float(np.max(np.abs(np.linalg.norm(latitude, axis=1) - 1.0))),
}
check_path = CHECK_ROOT / "sphere-distance-chart-checks.json"
save_json(checks, check_path)
generated_artifacts.append(check_path)
display_artifact(check_path)


## Concept frame

The central objects of this chapter can be read at three levels. First there is a synthetic level: points, lines, planes, spheres, tangencies, and intersections. Second there is an algebraic level: coordinates, matrices, determinants, ranks, signatures, and residuals. Third there is a metric or topological level when the chapter asks for length, area, angle, orientation, compactness, or connectedness. A standalone notebook has to keep those levels visible at the same time.

The diagrams below are therefore built from data rather than imported as fixed pictures. That choice matters. If the reader changes a parameter, the artifact changes with it and the check either continues to pass or reveals exactly which assumption was broken. This is especially useful in Berger's style of geometry, where an object is often introduced synthetically and then compared with an affine, projective, Euclidean, spherical, or hyperbolic model.

The chapter also rewards paying attention to degeneracy. Degenerate cases are not annoyances pushed to the margin; they are boundary points of a classification. A vanishing determinant, a point at infinity, a null direction, a tangent contact, or an ideal boundary can all explain why a theorem needs the hypotheses it has. The code keeps those cases close enough to see, without pretending that a numerical experiment is a proof.


## Worked example philosophy

The worked examples favor small complete constructions over large opaque demonstrations. Every object is named, every coordinate convention is stated, and every saved artifact has a filename that names the mathematical idea it illustrates. A reader should be able to rerun a cell, change one number, and predict which part of the visual will move.

The examples also separate representation from interpretation. A conic matrix is not itself a conic until we decide which projective chart we are viewing. A sphere drawn on a flat page is not intrinsically flat. A hyperbolic disk uses Euclidean pixels to represent non-Euclidean distance. These separations are the main source of both power and confusion, so they are made explicit before the applied lab.


## How to read the artifacts

The artifacts in this notebook should be read as a small laboratory record. The PNG files are durable views of the construction, but the nearby code is part of the lesson: it states the coordinate convention, the parameter values, and the invariant being tested. The JSON and CSV files are intentionally plain so that a reader can open them outside the notebook and see the same evidence without rerunning every cell.

When a visual compares several objects, read it from the invariant outward. In this chapter the invariant is usually one of these: unit-length constraints, angular distance, chart distortion, area element behavior, and preservation of circles under stereographic projection. Ask first what should stay fixed, then inspect which part of the figure changes. If the figure shows a family, the interesting information is often in the limiting member: a degenerate conic, a tangent contact, a boundary point, a null direction, or an approximation that is nearly smooth but still finite.

The saved checks do not replace proof. They play a different role. They protect the notebook from misleading pictures, record the numerical scale of residuals, and give the reader a concrete experiment to modify. A good modification changes the parameters while preserving the hypotheses; a better one also breaks a hypothesis and watches the check fail for a geometric reason.


## Applied lab

Project great circles and latitude circles stereographically, then compare spherical side lengths in a triangle with the Euclidean drawing of its chart.

For a stronger lab, change the parameters in two opposite directions: one that preserves the hypotheses and one that breaks them. The first change should keep the final checks small. The second should make a residual, signature, or visual feature fail in a meaningful way. That contrast is often where the real theorem becomes visible.


## Takeaways

- A chart is a measurement device, not a replacement for the sphere.
- Great circles are geodesics because they are plane sections through the origin.
- Stereographic projection keeps circles visible but distorts lengths and areas.
- Spherical triangles expose the difference between intrinsic and planar angle intuition.


In [ ]:
assert generated_artifacts, "the notebook should generate artifacts"
assert_artifacts(generated_artifacts, min_bytes=32)
final_sanity = {
    "artifact_count": len(generated_artifacts),
    "all_artifacts_exist": all(path.exists() for path in generated_artifacts),
    "artifact_root": ARTIFACT_ROOT.relative_to(BOOK_ROOT).as_posix(),
}
final_sanity
